# Pairwise LTR Scheduler — Cloud Experiment

Run on **Google Colab GPU** (Runtime → Change runtime type → T4).

1. Clone repo
2. Train pairwise ranker
3. Compare FCFS vs LTR vs Pairwise LTR

In [ ]:
# Clone from GitHub (update URL after you push)
!git clone https://github.com/YOUR_USERNAME/pairwise-ltr-scheduler.git
%cd pairwise-ltr-scheduler

In [ ]:
!pip install -q -r requirements.txt
import torch
print('GPU:', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU only')

In [ ]:
# Train the pairwise ranker (about 5-10 min on T4)
!python scripts/train_predictor.py --epochs 3 --train-samples 2000 --device cuda

In [ ]:
# Compare all scheduling policies
!python scripts/run_simulation.py --compare-all --checkpoint checkpoints/pairwise_ranker.pt --device cuda

In [ ]:
# Optional: test priority prompts
from src.priority import InferenceRequest, PriorityLevel
from src.scheduler import PairwiseLTRScheduler

scheduler = PairwiseLTRScheduler(batch_size=4)

scheduler.add_request(InferenceRequest('1', 'Write a 500 word essay', 500, PriorityLevel.LOW, rank_score=8.0))
scheduler.add_request(InferenceRequest('2', 'Yes or no?', 5, PriorityLevel.NORMAL, rank_score=1.0))
scheduler.add_request(InferenceRequest('3', 'Emergency summary needed', 20, PriorityLevel.HIGH, rank_score=3.0))

batch = scheduler.pick_next_batch()
print('Serve order:', [r.request_id for r in batch])